# Validation of `HyperelasticityFEMForceField`

This notebook validates SOFA's `HyperelasticityFEMForceField` against FEniCS, as presented in [IntroductionValidation.ipynb](IntroductionValidation.ipynb), on the cases in `validation/cases/HyperelasticityFEMForceField/`. These cases have no exact solution, so SOFA is compared with FEniCS only.

The force field takes its law from a linked material component. Every case runs three laws, on the deformation gradient $\mathbf{F} = \mathbf{I} + \nabla \mathbf{u}$, with $\mathbf{C} = \mathbf{F}^T \mathbf{F}$, $J = \det \mathbf{F}$ and $d$ the dimension:

| Law | SOFA component | Strain energy density $\psi$ |
|---|---|---|
| `SaintVenantKirchhoff` | `StVenantKirchhoffMaterial` | $\frac{\lambda}{2} (\operatorname{tr} \mathbf{E})^2 + \mu\, \mathbf{E} : \mathbf{E}$, with $\mathbf{E} = (\mathbf{C} - \mathbf{I}) / 2$ |
| `NeoHookean` | `NeoHookeanMaterial` | $\frac{\mu}{2} (\operatorname{tr} \mathbf{C} - d - 2 \ln J) + \frac{\lambda}{2} (\ln J)^2$ |
| `MooneyRivlin` | `MooneyRivlinMaterial` | $\mu_{10} (J^{-2/d} I_1 - d) + \mu_{01} (J^{-4/d} I_2 - \frac{d(d-1)}{2}) + \frac{\kappa}{2} (\ln J)^2$ |

where $I_1 = \operatorname{tr} \mathbf{C}$ and $I_2 = \frac{1}{2} \left( I_1^2 - \operatorname{tr}(\mathbf{C}^2) \right)$. The FEniCS scenes take the same $\psi$ from `common/materials.py`, converted to UFL by `common/fenics/materials.py`.

The load is applied in equal steps, each solved by Newton-Raphson. In SOFA, `LoadRamp` (`common/sofa/controllers.py`) scales the load at the start of each step.

On quadrilateral and hexahedron elements, the integrand of a hyperelastic law is not a polynomial, so the result depends on the quadrature points. FEniCS therefore integrates at SOFA's points: a 3-point rule on quadrilaterals, and a $2 \times 2 \times 2$ Gauss rule on hexahedra.

## Running the Cases

The cells below load the suite's runner and prepare the figures. Run them once before any case: every case cell below then runs on its own, in any order.

In [ ]:
# Make the plugin's packages importable from validation/docs
import sys
sys.path.insert(0, "../..")

# Hide the runner's printed report; figures still show
import os
sys.stdout = open(os.devnull, "w")

# Load the suite's runner
from validation.run import run

# For showing figures inline
%matplotlib inline

`side_by_side` shows figures in one row. `show_laws` shows a row per law, with the case's elements side by side.

In [ ]:
import base64
import io
from IPython.display import HTML, display

def side_by_side(*figures):
    images = []
    for figure in figures:
        buffer = io.BytesIO()
        figure.savefig(buffer, format="png")
        encoded = base64.b64encode(buffer.getvalue()).decode()
        images.append(f'<img src="data:image/png;base64,{encoded}" style="width: {100 / len(figures)}%">')
    display(HTML(f'<div style="display: flex">{"".join(images)}</div>'))

LAWS = ["SaintVenantKirchhoff", "NeoHookean", "MooneyRivlin"]

def show_laws(study, *elements):
    for law in LAWS:
        side_by_side(*(study.plots[f"{law}_{element}"] for element in elements))

## Triangle and Quadrilateral Elements on a 2D Beam

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$. Each case is run on quadrilateral (`quad`) and triangle (`tri`) elements.

### Distributed Transverse Traction

The beam is clamped on its left side, and its top side carries a uniform downward traction $q$, applied in 5 steps.

- `left` ($x = 0$): $\mathbf{u} = 0$
- `top` ($y = W$): $\mathbf{t} = (0, -q)$
- `right` ($x = L$) and `bottom` ($y = 0$): $\mathbf{t} = 0$

In [ ]:
# Case: 2D cantilever under a distributed transverse traction
study = run("../cases/HyperelasticityFEMForceField/2D/DistributedTransverseTraction")
# a row per law, quad on the left, tri on the right
show_laws(study, "quad", "tri")

### Three-Point Bending

The beam rests on two rollers at $(0, 0)$ and $(L, 0)$, and a point force $P$, applied in 5 steps, pushes down on the middle of its top side. The bottom of the midspan is held along $x$, which stops the beam from sliding without breaking its symmetry.

- rollers $(0, 0)$ and $(L, 0)$: $u_y = 0$
- node $(L/2, 0)$: $u_x = 0$
- node $(L/2, W)$: force $(0, -P)$
- all sides: $\mathbf{t} = 0$

In [ ]:
# Case: 2D beam in three-point bending
study = run("../cases/HyperelasticityFEMForceField/2D/ThreePointBending")
# a row per law, quad on the left, tri on the right
show_laws(study, "quad", "tri")

## Tetrahedron and Hexahedron Elements on a 3D Beam

The beam starts at the origin, with length $L$ along $x$, width $W$ along $y$ and height $H$ along $z$. Each case is run on hexahedron (`hexa`) and tetrahedron (`tet`) elements.

### Distributed Transverse Traction

The beam is clamped on its left face, and its top face carries a uniform downward traction $q$, applied in 5 steps.

- `left` ($x = 0$): $\mathbf{u} = 0$
- `top` ($y = W$): $\mathbf{t} = (0, -q, 0)$
- the other faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D cantilever under a distributed transverse traction
study = run("../cases/HyperelasticityFEMForceField/3D/DistributedTransverseTraction")
# a row per law, hexa on the left, tet on the right
show_laws(study, "hexa", "tet")

### Three-Point Bending

The beam rests on a pinned line at $x = 0$ and on a roller line at $x = L$, both on its bottom face and along $z$. A force $P$, applied in 5 steps, pushes down on each node of the line across the middle of its top face.

- line $(0, 0, z)$: $u_x = u_y = 0$
- line $(L, 0, z)$: $u_y = 0$
- node $(0, 0, 0)$: $u_z = 0$
- line $(L/2, W, z)$: force $(0, -P, 0)$ on each node
- all faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D beam in three-point bending
study = run("../cases/HyperelasticityFEMForceField/3D/ThreePointBending")
# a row per law, hexa on the left, tet on the right
show_laws(study, "hexa", "tet")

## Tetrahedron Elements on a 3D Circular Beam

The beam is a cylinder of radius $R$ and length $L$ along $x$, its left face centred at the origin. `mesh.py` in the case folder meshes it with tetrahedra (`tet`) using [gmsh](https://gmsh.info), and SOFA and FEniCS both read that mesh file. The left face is clamped, and the lateral surface is traction-free.

### Torsion

The right face carries a torque $T$ about the axis, applied in 5 steps, as the traction of a circular shaft in torsion: tangential, with magnitude $T r / J$ at the distance $r$ from the axis, where $J = \pi R^4 / 2$ is the polar moment of the section.

- left face ($x = 0$): $\mathbf{u} = 0$
- right face ($x = L$): $\mathbf{t} = \frac{T}{J}\,(0, -z, y)$

In [ ]:
# Case: 3D circular beam in torsion
study = run("../cases/HyperelasticityFEMForceField/3D/CircularTorsion")
# a row per law
show_laws(study, "tet")

## Tetrahedron and Hexahedron Elements on a 3D Cruciform

The cruciform is a plate of thickness $t$ in the $xy$ plane, centred at the origin: a cross of four arms of width $2 w_0$, reaching $\pm l_0$ along $x$ and $y$, with $w_0 / l_0 = 1/3$. Its geometry and loading follow the cruciform biaxial test of Nolan and McGarry [[1]](#ref1), with square corners, and with the arms pulled by a traction rather than by clamps. The material parameters derive from their compressible Neo-Hookean values, $\mu = 7$ and $\kappa = 60$: $E$ and $\nu$ for `SaintVenantKirchhoff` and `NeoHookean`, and $\mu_{10} = \mu_{01} = \mu / 4$ for `MooneyRivlin`.

`mesh.py` in the case folder meshes it with hexahedra (`hexa`) and tetrahedra (`tet`) using [gmsh](https://gmsh.info), and SOFA and FEniCS both read those mesh files.

### Biaxial Traction

Each arm end carries a uniform traction $\sigma$ pointing outwards, applied in 5 steps. The tractions balance, so the rigid-body motions are removed where the equibiaxial symmetry already sets the displacement to zero.

- arm ends ($x = \pm l_0$): $\mathbf{t} = (\pm \sigma, 0, 0)$
- arm ends ($y = \pm l_0$): $\mathbf{t} = (0, \pm \sigma, 0)$
- centre $(0, 0, 0)$: $\mathbf{u} = 0$
- line $(x, 0, 0)$: $u_y = u_z = 0$
- line $(0, y, 0)$: $u_x = u_z = 0$
- the other faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D cruciform under biaxial traction
study = run("../cases/HyperelasticityFEMForceField/3D/CruciformBiaxialTraction")
# a row per law, hexa on the left, tet on the right
show_laws(study, "hexa", "tet")

## References

<a id="ref1"></a>[1] D. R. Nolan and J. P. McGarry, "On the correct interpretation of measured force and calculation of material stress in biaxial tests", *Journal of the Mechanical Behavior of Biomedical Materials*, 2016. https://doi.org/10.1016/j.jmbbm.2015.08.019